# Урок 8.3. Поиск разбиений: точный перебор, гистограммы и пропуски

**Интерактивная версия:** `lessons/lesson_8_3/web/index.html`

1. Точный перебор с накопленными суммами — своими руками.
2. Гистограмма и вычитание гистограмм.
3. Точность в зависимости от числа корзин.
4. Скорость `tree_method` в XGBoost и LightGBM.
5. Пропуски: направление по умолчанию; сравнение с заменой.

In [1]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "python" / "gbcourse").is_dir())
sys.path.insert(0, str(ROOT / "shared" / "python"))

import json

import numpy as np
import pandas as pd
from gbcourse import datasets, GBRegressor, RegressionTree
from gbcourse.tree import make_bins, bin_codes

## 1. Точный перебор

In [2]:
X, y = datasets.friedman1(n=3000, noise=1.0, seed=84)
g = y.mean() - y
h = np.ones_like(g)
lam = 1.0
S = lambda G, H: G * G / (H + lam)


def best_exact(x, g, h):
    order = np.argsort(x, kind="stable")
    xs, GL, HL = x[order], np.cumsum(g[order])[:-1], np.cumsum(h[order])[:-1]
    G, H = g.sum(), h.sum()
    gain = 0.5 * (S(GL, HL) + S(G - GL, H - HL) - S(G, H))
    gain[xs[:-1] == xs[1:]] = -np.inf  # между равными значениями порога нет
    k = int(np.argmax(gain))
    return (xs[k] + xs[k + 1]) / 2, gain[k]


for j in range(X.shape[1]):
    thr, gain = best_exact(X[:, j], g, h)
    print(f"признак {j}: лучший порог {thr:.4f}, gain {gain:9.2f}")
tree = RegressionTree(max_depth=1, reg_lambda=lam).fit(X, g)
print("RegressionTree:", tree.describe().splitlines()[0])

признак 0: лучший порог 0.2790, gain   6053.04
признак 1: лучший порог 0.3135, gain   5875.39
признак 2: лучший порог 0.8017, gain   1140.95
признак 3: лучший порог 0.5420, gain   9444.95
признак 4: лучший порог 0.5749, gain   2759.37
признак 5: лучший порог 0.0414, gain    112.86
признак 6: лучший порог 0.0092, gain     37.29
признак 7: лучший порог 0.9869, gain     78.14
признак 8: лучший порог 0.0664, gain     89.11
признак 9: лучший порог 0.9556, gain    149.95
RegressionTree: ├─ если x3 ≤ 0.542  (n = 3000, выигрыш = 9444.953)


## 2. Гистограмма и вычитание гистограмм

In [3]:
bins = make_bins(X, 32)
codes = bin_codes(X, bins)
j = 3
Gb = np.bincount(codes[:, j], weights=g, minlength=len(bins[j]) + 1)
Hb = np.bincount(codes[:, j], weights=h, minlength=len(bins[j]) + 1)
GL, HL = np.cumsum(Gb)[:-1], np.cumsum(Hb)[:-1]
gain = 0.5 * (S(GL, HL) + S(g.sum() - GL, h.sum() - HL) - S(g.sum(), h.sum()))
k = int(np.argmax(gain))
print(f"признак {j}: 32 корзины → порог {bins[j][k]:.4f}, gain {gain[k]:.2f}; точно — {best_exact(X[:, j], g, h)}")

left = X[:, 0] <= 0.5  # какое-нибудь разбиение узла
Gl = np.bincount(codes[left, j], weights=g[left], minlength=len(Gb))
Gr_direct = np.bincount(codes[~left, j], weights=g[~left], minlength=len(Gb))
print("правая гистограмма = родитель − левая:", np.allclose(Gb - Gl, Gr_direct))

признак 3: 32 корзины → порог 0.5344, gain 9405.57; точно — (np.float64(0.5419605685165152), np.float64(9444.952824442998))
правая гистограмма = родитель − левая: True


## 3. Точность в зависимости от числа корзин

In [4]:
a, b, c, d = datasets.train_test_split(X, y, test_size=0.3, seed=0)
rows = []
for mb in (None, 4, 8, 16, 32, 64, 255):
    m = GBRegressor(mode="newton", n_estimators=150, learning_rate=0.1, max_depth=4, reg_lambda=1, max_bins=mb).fit(a, c, eval_set=(b, d))
    rows.append({"корзин": "точно" if mb is None else mb, "лучшая ½MSE": round(min(m.history_["eval"]), 4)})
pd.DataFrame(rows)

,корзин,лучшая ½MSE
0,точно,0.8212
1,4,2.0170
2,8,1.0398
3,16,0.8208
4,32,0.7838
5,64,0.7876
6,255,0.7956


## 4. Скорость в библиотеках

Время зависит от машины; важно соотношение.

In [5]:
import lightgbm as lgb
import xgboost as xgb

Xb, yb = datasets.friedman1(n=200_000, noise=1.0, seed=85)
times = {}
for tm in ("exact", "approx", "hist"):
    t0 = time.perf_counter()
    xgb.XGBRegressor(n_estimators=100, max_depth=6, tree_method=tm, n_jobs=4).fit(Xb, yb)
    times[f"XGBoost {tm}"] = time.perf_counter() - t0
t0 = time.perf_counter()
lgb.LGBMRegressor(n_estimators=100, num_leaves=63, max_depth=6, n_jobs=4, verbose=-1).fit(Xb, yb)
times["LightGBM"] = time.perf_counter() - t0
for k_, v in times.items():
    print(f"{k_:16s} {v:6.2f} с")

XGBoost exact     10.13 с
XGBoost approx     1.02 с
XGBoost hist       0.46 с
LightGBM           0.49 с


## 5. Пропуски

### Направление по умолчанию в XGBoost, если в обучении пропусков не было

In [6]:
Xs, ys = datasets.friedman1(n=500, noise=1.0, seed=87)
for tm in ("exact", "hist", "approx"):
    booster = xgb.XGBRegressor(n_estimators=3, max_depth=3, tree_method=tm).fit(Xs, ys).get_booster()
    left = total = 0
    stack = [json.loads(t) for t in booster.get_dump(dump_format="json")]
    while stack:
        nd = stack.pop()
        if "leaf" in nd:
            continue
        total += 1
        left += nd["missing"] == nd["children"][0]["nodeid"]
        stack.extend(nd["children"])
    print(f"{tm:6s}: пропуски влево в {left} из {total} узлов")

exact : пропуски влево в 21 из 21 узлов
hist  : пропуски влево в 0 из 21 узлов
approx: пропуски влево в 0 из 21 узлов


### Встроенная обработка против замены

30% пропусков в сильном признаке $x_3$ — только у объектов с $y$ выше 70-го перцентиля (с вероятностью 0.7).

In [7]:
res = {"NaN (направление по умолчанию)": [], "замена средним": [], "замена на −1": []}
for s in range(3):
    Xm, ym = datasets.friedman1(n=1500, noise=1.0, seed=86 + s)
    rng = np.random.default_rng(s)
    miss = rng.random(len(ym)) < np.where(ym > np.quantile(ym, 0.7), 0.7, 0.0)
    Xm = Xm.copy()
    Xm[miss, 3] = np.nan
    a, b, c, d = datasets.train_test_split(Xm, ym, test_size=0.3, seed=s)
    for name in res:
        a2, b2 = a.copy(), b.copy()
        if name != "NaN (направление по умолчанию)":
            fill = np.nanmean(a[:, 3]) if name == "замена средним" else -1.0
            a2[np.isnan(a2[:, 3]), 3] = fill
            b2[np.isnan(b2[:, 3]), 3] = fill
        m = GBRegressor(mode="newton", n_estimators=200, learning_rate=0.1, max_depth=4, reg_lambda=1).fit(a2, c, eval_set=(b2, d))
        res[name].append(min(m.history_["eval"]))
pd.Series({k_: np.mean(v) for k_, v in res.items()}, name="лучшая ½MSE").round(4)

NaN (направление по умолчанию)    1.4230
замена средним                    1.4709
замена на −1                      1.4086
Name: лучшая ½MSE, dtype: float64

## Упражнения

1. Реализуйте поиск лучшего порога по гистограмме с пропусками (два варианта направления) и сверьте с `RegressionTree(max_bins=32)`.
2. Для признака $x_0$ постройте долю потерянного выигрыша лучшего порога в зависимости от числа корзин (2…256).

Решения: `python lessons/lesson_8_3/exercises/solutions.py`.